# GLiClass 학습 알고리즘 직접 구현하기 (NumPy → PyTorch 검증)

목표: GLiClass가 학습할 때 안에서 일어나는 일을 NumPy로 직접 계산하고, PyTorch(실제 학습에 쓰는 도구)와 값이 똑같은지 확인한다.

| 단계 | 하는 일 | 이 노트북 섹션 |
|---|---|---|
| 1 | 읽기: 라벨들과 로그를 한 번에 인코더에 넣음 | (모델 내부, 생략) |
| 2 | 점수: 라벨마다 로그와 얼마나 맞는지 숫자 하나 | 1 |
| 3 | 확률: sigmoid로 0~1로 바꿈 (softmax와 비교) | 1 |
| 4 | 채점: BCE → focal loss | 2, 3 |
| 5 | 고치기: 기울기 계산 후 가중치를 조금 이동 | 4, 5 |
| + | 잊지 않기: EWC 벌점 | 6 |
| + | 입력 길이: 실제 토큰 수 측정 | 7 |

예시 상황: 로그 = "IAM 사용자 목록 조회 3번, 권한 상세 조회 1번", 라벨 A = 계정 목록 조회(Discovery), B = 비밀값 훔치기(Credential Access), C = 정상 업무(Benign), 정답 = A

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

np.set_printoptions(precision=4, suppress=True)
labels = ["A 계정 목록 조회", "B 비밀값 훔치기", "C 정상 업무"]

## 1. 점수 → 확률: sigmoid vs softmax

모델이 낸 점수(logit)가 A=0.5, B=1.0, C=2.0 이라고 하자. (학습 전이라 정상 C가 제일 높음 = 사전 실험 상태)

- sigmoid: 라벨마다 따로 `1 / (1 + e^-x)`. 서로 영향 없음. 합이 1일 필요 없음.
- softmax: `e^x / 모든 e^x의 합`. 확률 1을 나눠 가짐. 하나 오르면 나머지가 내려감.

In [ ]:
logits = np.array([0.5, 1.0, 2.0])

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def softmax(x):
    e = np.exp(x - x.max())          # 큰 수 넘침 방지용으로 최댓값을 빼도 결과는 같음
    return e / e.sum()

print("sigmoid:", sigmoid(logits), " 합 =", sigmoid(logits).sum().round(3))
print("softmax:", softmax(logits), " 합 =", softmax(logits).sum().round(3))

# PyTorch와 비교
t = torch.tensor(logits)
print("torch 일치(sigmoid):", np.allclose(sigmoid(logits), torch.sigmoid(t).numpy()))
print("torch 일치(softmax):", np.allclose(softmax(logits), torch.softmax(t, 0).numpy()))

### 정답이 여러 개일 때 (T1078 같은 경우)

탈취한 계정으로 로그인하는 행동은 초기 침투, 지속성, 권한 상승, 은닉 4개 전술에 동시에 속한다. 네 라벨 모두 점수가 높게(3.0) 나왔다고 해보자.

In [ ]:
multi = np.array([3.0, 3.0, 3.0, 3.0, -2.0])   # 정답 4개 + 오답 1개
print("sigmoid:", sigmoid(multi))   # 4개 모두 0.95 -> '전부 맞다'고 말할 수 있음
print("softmax:", softmax(multi))   # 4개가 0.25씩 나눠 가짐 -> 아무것도 확신 못 함

## 2. 채점: BCE (이진 교차 엔트로피)

라벨마다 "예(1)/아니오(0)" 문제로 보고 벌점을 매긴다.

`벌점 = -[ y * log(p) + (1 - y) * log(1 - p) ]`

- 정답(y=1)인데 p가 작으면 벌점 큼
- 오답(y=0)인데 p가 크면 벌점 큼

In [ ]:
y = np.array([1.0, 0.0, 0.0])   # 정답지: A만 정답
p = sigmoid(logits)

def bce(p, y):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

per_label = bce(p, y)
for name, pi, yi, li in zip(labels, p, y, per_label):
    print(f"{name}: 정답={int(yi)}, 확률={pi:.2f}, 벌점={li:.2f}")
print("평균 벌점:", per_label.mean().round(4))

torch_bce = F.binary_cross_entropy_with_logits(torch.tensor(logits), torch.tensor(y), reduction="none").numpy()
print("torch 일치:", np.allclose(per_label, torch_bce))

## 3. 채점 업그레이드: focal loss (GLiClass가 실제로 쓰는 것)

우리 상황: 라벨이 60개 넘는데 정답은 1개. 쉬운 오답 59개의 작은 벌점이 쌓이면 진짜 어려운 문제가 묻힌다.

focal loss = BCE에 두 가지를 곱함
- `(1 - p_t)^γ` : 이미 잘 맞힌 문제(p_t가 1에 가까움)는 벌점을 거의 0으로 줄임. γ=2가 기본.
- `α_t` : 정답 라벨과 오답 라벨의 비중 조절. GLiClass 기본 α=0.5 (반반).

여기서 `p_t` = "정답 쪽으로 준 확률" (정답이면 p, 오답이면 1-p)

아래 함수는 GLiClass 저장소 `gliclass/loss_functions.py`의 `focal_loss_with_logits`를 NumPy로 옮긴 것.

In [ ]:
def focal_loss(logits, y, alpha=0.5, gamma=2.0):
    p = sigmoid(logits)
    loss = bce(p, y)
    p_t = p * y + (1 - p) * (1 - y)           # 정답 쪽으로 준 확률
    loss = loss * (1 - p_t) ** gamma          # 쉬운 문제 벌점 줄이기
    alpha_t = alpha * y + (1 - alpha) * (1 - y)
    return alpha_t * loss

print("BCE  :", bce(sigmoid(logits), y))
print("focal:", focal_loss(logits, y))

# 쉬운 문제 vs 어려운 문제에서 차이 보기
easy = np.array([4.0]); hard = np.array([-1.0]); one = np.array([1.0])
print("\n정답인데 이미 잘 맞힌 경우(쉬움)  BCE=%.4f  focal=%.4f" % (bce(sigmoid(easy), one)[0], focal_loss(easy, one)[0]))
print("정답인데 틀리게 본 경우(어려움)  BCE=%.4f  focal=%.4f" % (bce(sigmoid(hard), one)[0], focal_loss(hard, one)[0]))
print("-> focal은 쉬운 문제 벌점을 크게 깎고, 어려운 문제 벌점은 상대적으로 남긴다")

In [ ]:
# PyTorch로 같은 식을 짜서 일치 확인 (GLiClass 코드와 같은 구조)
def focal_torch(inputs, targets, alpha=0.5, gamma=2.0):
    p = torch.sigmoid(inputs)
    loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction="none")
    p_t = p * targets + (1 - p) * (1 - targets)
    loss = loss * ((1 - p_t) ** gamma)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    return alpha_t * loss

print("torch 일치:", np.allclose(focal_loss(logits, y), focal_torch(torch.tensor(logits), torch.tensor(y)).numpy()))

# Colab에서 gliclass를 설치했다면 진짜 GLiClass 함수와도 비교 가능
try:
    from gliclass.loss_functions import focal_loss_with_logits
    real = focal_loss_with_logits(torch.tensor(logits), torch.tensor(y), alpha=0.5, gamma=2, reduction="none").numpy()
    print("GLiClass 원본과 일치:", np.allclose(focal_loss(logits, y), real))
except ImportError:
    print("(gliclass 미설치: !pip install gliclass 후 다시 실행하면 원본과도 비교됨)")

## 4. 고치기: 기울기(gradient)

질문: "점수(logit)를 살짝 올리면 벌점이 늘어나나 줄어드나?" 이 방향과 크기가 기울기.

sigmoid + BCE 조합은 기울기가 아주 깔끔하다: **`기울기 = p - y`**
- A: 0.62 - 1 = -0.38 → 음수 → 점수를 올려야 벌점이 줄어듦
- C: 0.88 - 0 = +0.88 → 양수 → 점수를 내려야 벌점이 줄어듦

PyTorch는 이걸 자동으로 계산한다(autograd). 손으로 구한 값과 같은지 확인.

In [ ]:
grad_np = sigmoid(logits) - y
print("NumPy 손계산 기울기:", grad_np)

t = torch.tensor(logits, requires_grad=True)
F.binary_cross_entropy_with_logits(t, torch.tensor(y), reduction="sum").backward()
print("PyTorch 자동 기울기:", t.grad.numpy())
print("일치:", np.allclose(grad_np, t.grad.numpy()))

## 5. 작은 모델로 학습 루프 돌려보기

실제 GLiClass는 숫자 3,270만 개지만, 원리는 같다. 여기서는 숫자 몇 개짜리 장난감 모델로 해본다.

- 로그 벡터 `x` (인코더가 로그를 읽고 만든 숫자 4개라고 가정)
- 라벨 벡터 `L` (라벨 문장을 읽고 만든 숫자, 라벨당 4개)
- 학습할 가중치 `W` (4x4 행렬)
- 점수 = `L @ (W @ x)` : 라벨 벡터와 '변환된 로그 벡터'의 내적 = 얼마나 비슷한가

학습 = 읽기 → 점수 → 확률 → 채점 → 기울기 → W를 조금 이동, 반복

In [ ]:
rng = np.random.default_rng(0)
x = rng.normal(size=4)                 # 로그 벡터
L = rng.normal(size=(3, 4))            # 라벨 A, B, C 벡터
W0 = np.eye(4) * 0.3 + rng.normal(scale=0.1, size=(4, 4))   # 학습 전 가중치
y = np.array([1.0, 0.0, 0.0])
lr = 0.1                                # 학습률 (실제 GLiClass 파인튜닝은 0.00001 수준)

W = W0.copy()
for step in range(31):
    z = W @ x                           # 변환된 로그 벡터
    s = L @ z                           # 2단계: 점수
    p = sigmoid(s)                      # 3단계: 확률
    loss = bce(p, y).mean()             # 4단계: 채점
    g_s = (p - y) / len(y)              # 5단계: 점수에 대한 기울기
    g_W = np.outer(L.T @ g_s, x)        #        W에 대한 기울기 (연쇄법칙)
    W -= lr * g_W                       #        조금 이동
    if step % 10 == 0:
        print(f"step {step:2d}  벌점 {loss:.3f}  확률 A/B/C = {p.round(2)}")

In [ ]:
# 같은 학습을 PyTorch로: 기울기를 직접 안 짜도 autograd가 계산
Wt = torch.tensor(W0, requires_grad=True)
opt = torch.optim.SGD([Wt], lr=lr)
xt, Lt, yt = map(torch.tensor, (x, L, y))
for step in range(31):
    s = Lt @ (Wt @ xt)
    loss = F.binary_cross_entropy_with_logits(s, yt)
    opt.zero_grad(); loss.backward(); opt.step()

print("NumPy 학습 결과 W와 PyTorch 결과 W 일치:", np.allclose(W, Wt.detach().numpy()))
print("-> 실제 GLiClass 학습은 이 SGD 자리에 AdamW, BCE 자리에 focal loss를 쓴다")

## 6. 잊지 않기: EWC 벌점

문제: 새 과목(로그)을 공부하다 원래 실력(일반 텍스트 분류)을 잊는다.

EWC 아이디어: 원래 과목에 **중요했던 숫자**는 많이 못 움직이게 벌점을 준다.

`EWC 벌점 = λ × Σ F_i × (θ_i - θ*_i)²`
- `θ*` : 학습 전 원래 가중치
- `θ` : 지금 가중치
- `F` : 중요도(피셔 정보). 원래 데이터로 기울기를 구해 제곱해서 평균 낸 값 → 기울기가 컸던 숫자 = 결과에 큰 영향 = 중요
- `λ` : 얼마나 세게 붙잡을지

GLiClass 저장소 `gliclass/training.py`의 `ewc_loss`와 같은 식. 아래에서 원래 과목(옛 로그 x_old, 정답 B)과 새 과목(x, 정답 A)을 두고 EWC를 끈/켠 경우를 비교한다.

In [ ]:
x_old = rng.normal(size=4); y_old = np.array([0.0, 1.0, 0.0])

# (1) 원래 과목을 먼저 학습시켜 '기존 모델'을 만든다
W_old = W0.copy()
for _ in range(200):
    p = sigmoid(L @ (W_old @ x_old))
    W_old -= 0.1 * np.outer(L.T @ ((p - y_old) / 3), x_old)

# (2) 중요도 F: 원래 과목 데이터에서 기울기 제곱
p = sigmoid(L @ (W_old @ x_old))
F_imp = np.outer(L.T @ ((p - y_old) / 3), x_old) ** 2 + 1e-3
F_imp = F_imp / F_imp.max()            # GLiClass도 최댓값으로 정규화(normalize_fisher)

def train_new(use_ewc, lam=5.0, steps=200):
    Wn = W_old.copy()
    for _ in range(steps):
        p = sigmoid(L @ (Wn @ x))
        g = np.outer(L.T @ ((p - y) / 3), x)
        if use_ewc:
            g += lam * 2 * F_imp * (Wn - W_old)    # EWC 벌점의 기울기
        Wn -= 0.1 * g
    return Wn

rows = [("기존 모델 (학습 전)", W_old), ("EWC 없음", train_new(False))]
rows += [(f"EWC λ={lam}", train_new(True, lam)) for lam in [0.2, 1.0, 5.0]]
for name, Wn in rows:
    new_acc = sigmoid(L @ (Wn @ x))[0]
    old_acc = sigmoid(L @ (Wn @ x_old))[1]
    print(f"{name:16s}  새 과목(로그) 정답확률 {new_acc:.2f} | 원래 과목 정답확률 {old_acc:.2f}")
print("-> λ가 클수록 원래 실력은 지키지만 새 과목은 덜 배운다. 우리 실험에서 λ를 고르는 게 이 균형 찾기")

## 7. 입력 길이: 실제 토큰 수 재기

확인된 edge 모델 정보: 인코더 ModernBERT, 10층, 크기 384, **최대 위치 7,999 토큰**, 라벨이 앞(prompt_first=True).

GLiClass 입력 순서: `[라벨들] <<SEP>> [로그 문장] [예시들]` → 넘치면 **뒤에서부터 잘림** → 예시가 먼저, 그다음 로그가 잘린다.

CSV 업로드 없이 바로 실행 가능 (라벨 83개를 노트북 안에 넣어둠). 파일을 올리면 그 파일을 우선 사용.

In [ ]:
LABELS_V19 = """node_id|level|name|desc
TA0001|top|Initial Access|getting first access to the cloud account with valid or trusted credentials
TA0002|top|Execution|running commands or code on cloud instances or serverless functions
TA0003|top|Persistence|keeping future access by adding credentials, accounts, triggers or images
TA0004|top|Privilege Escalation|gaining broader permissions by changing policies or assuming stronger roles
TA0005|top|Stealth|hiding activity by using unusual regions or legitimate-looking accounts
TA0112|top|Defense Impairment|weakening logging, firewalls, security services or compute safeguards
TA0006|top|Credential Access|obtaining passwords, secrets, keys or instance credentials
TA0007|top|Discovery|listing or describing accounts, permissions, resources and security settings
TA0008|top|Lateral Movement|connecting from the cloud control plane into running instances
TA0009|top|Collection|gathering stored data from cloud storage before taking it out
TA0010|top|Exfiltration|moving data or snapshots to an account or network outside the owner's control
TA0040|top|Impact|deleting, encrypting, stopping or abusing resources at the owner's expense
BENIGN|top|Benign|routine work by authorized operators, pipelines or service roles
T1078|mid|Valid Accounts|using credentials of an account that already exists
T1078.004|leaf|Cloud Accounts|signing in or calling APIs with an existing cloud account's credentials
T1199|mid|Trusted Relationship|entering through a role trusted by another organization's account
T1651|mid|Cloud Administration Command|running commands on instances through a cloud management service
T1648|mid|Serverless Execution|creating or updating serverless functions so they run new code
T1098|mid|Account Manipulation|changing credentials or permissions of an existing identity
T1098.001|leaf|Additional Cloud Credentials|issuing new access keys, login passwords or key pairs for an identity
T1098.003|leaf|Additional Cloud Roles|attaching or adding permission policies to a user or role
T1098.004|leaf|SSH Authorized Keys|adding ssh public keys that allow later logins to instances
T1136|mid|Create Account|creating a new identity in the account
T1136.003|leaf|Cloud Account|creating a new IAM user identity
T1546|mid|Event Triggered Execution|creating rules or triggers that run code automatically on events
T1525|mid|Implant Internal Image|pushing a modified container or machine image to a registry
T1548|mid|Abuse Elevation Control Mechanism|using built-in mechanisms that grant higher privileges temporarily
T1548.005|leaf|Temporary Elevated Cloud Access|assuming a more privileged role or passing a role to a service
T1535|mid|Unused/Unsupported Cloud Regions|creating or running resources in regions the account does not normally use
T1685|mid|Disable or Modify Tools|turning off or changing logging and security tools
T1685.002|leaf|Disable or Modify Cloud Log|stopping, deleting or narrowing cloud audit and network logging
T1685.SEC|leaf|Disable Security Services|disabling or reconfiguring cloud threat detection and security services
T1686|mid|Disable or Modify System Firewall|changing firewall rules to allow traffic that was blocked
T1686.001|leaf|Cloud Firewall|opening security groups or network ACLs to wide network access
T1578|mid|Modify Cloud Compute Infrastructure|creating, deleting or changing compute resources to avoid defenses
T1578.001|leaf|Create Snapshot|creating snapshots of volumes or databases
T1578.002|leaf|Create Cloud Instance|launching new compute instances
T1578.003|leaf|Delete Cloud Instance|terminating compute instances
T1578.004|leaf|Revert Cloud Instance|restoring instances from snapshots to erase changes
T1578.005|leaf|Modify Cloud Compute Configurations|changing account-level compute settings such as quotas
T1666|mid|Modify Cloud Resource Hierarchy|leaving or changing the cloud organization or account structure
T1110|mid|Brute Force|trying many passwords until a sign-in succeeds
T1110.003|leaf|Password Spraying|trying a few common passwords against many accounts
T1110.004|leaf|Credential Stuffing|trying leaked username and password pairs
T1555|mid|Credentials from Password Stores|reading credentials kept in a password or secrets store
T1555.006|leaf|Cloud Secrets Management Stores|retrieving many secrets or decrypted parameters from a secrets store
T1552|mid|Unsecured Credentials|finding credentials stored in insecure places
T1552.005|leaf|Cloud Instance Metadata API|using instance role credentials taken from the metadata service
T1552.GEN|leaf|Credentials in Instance Settings|reading passwords or secrets stored in instance settings
T1528|mid|Steal Application Access Token|obtaining application tokens to act as that application
T1606|mid|Forge Web Credentials|creating credentials that let the attacker sign in as others
T1606.002|leaf|SAML Tokens|adding or changing SAML providers to forge sign-ins
T1087|mid|Account Discovery|listing accounts that exist
T1087.004|leaf|Cloud Account|listing IAM users, roles and their details
T1580|mid|Cloud Infrastructure Discovery|describing instances, volumes, snapshots and networks
T1526|mid|Cloud Service Discovery|enumerating which cloud services are in use across the account
T1619|mid|Cloud Storage Object Discovery|listing objects inside storage buckets
T1654|mid|Log Enumeration|reading audit log settings or searching past events
T1069|mid|Permission Groups Discovery|finding groups and permissions assigned to identities
T1069.003|leaf|Cloud Groups|listing groups and policies attached to identities
T1201|mid|Password Policy Discovery|reading the account password policy
T1518|mid|Software Discovery|checking which software and services are installed or enabled
T1518.001|leaf|Security Software Discovery|checking which security services are enabled
T1021|mid|Remote Services|logging in to systems through remote access services
T1021.008|leaf|Direct Cloud VM Connections|connecting to instances through cloud remote access services
T1530|mid|Data from Cloud Storage|reading many objects from storage buckets
T1074|mid|Data Staged|copying data to one place before taking it out
T1537|mid|Transfer Data to Cloud Account|sharing snapshots, images or buckets with an external account
T1048|mid|Exfiltration Over Alternative Protocol|opening a network path to move data outside
T1531|mid|Account Access Removal|deleting or changing credentials so owners lose access
T1485|mid|Data Destruction|destroying stored data or resources
T1485.GEN|leaf|Delete Data or Resources|deleting buckets, objects, databases or machine images
T1485.001|leaf|Lifecycle-Triggered Deletion|setting storage lifecycle rules that delete data
T1486|mid|Data Encrypted for Impact|re-encrypting stored data with a key the owner does not control
T1490|mid|Inhibit System Recovery|deleting snapshots and backups needed for recovery
T1496|mid|Resource Hijacking|using the owner's resources for the attacker's benefit
T1496.001|leaf|Compute Hijacking|launching many or large instances for the attacker's workloads
T1496.004|leaf|Cloud Service Hijacking|calling paid cloud services such as AI models or email at scale
T1489|mid|Service Stop|stopping or deleting running services
BEN.DEPLOY|mid|Benign Deployment|building, pushing or rolling out software through an authorized pipeline
BEN.MONITOR|mid|Benign Monitoring|routine checks, listing or describing resources by operators or tools
BEN.AUTO|mid|Benign Automation|scheduled jobs and service roles doing their configured tasks
BEN.OPS|mid|Benign Operations|operators scaling, maintaining or configuring resources as planned"""

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("knowledgator/gliclass-edge-v3.0")
LIMIT = 7999

import io, os
# CSV 파일이 있으면 그걸 쓰고, 없으면 바로 위 셀에 넣어둔 라벨 목록(labels/attack_labels_v19.csv와 같은 83개)을 씀
if os.path.exists("attack_labels_v19.csv"):
    lab = pd.read_csv("attack_labels_v19.csv")
else:
    lab = pd.read_csv(io.StringIO(LABELS_V19), sep="|")
print("라벨 수:", len(lab))
lab["text"] = lab["name"] + ": " + lab["desc"]
lab["tokens"] = lab["text"].apply(lambda s: len(tok(s, add_special_tokens=False)["input_ids"]) + 1)  # +1 = <<LABEL>> 토큰

print(lab.groupby("level")["tokens"].agg(["count", "sum", "mean"]).round(1))
print("\n라벨 전체를 한 번에 넣을 때:", lab["tokens"].sum(), "토큰 / 한계", LIMIT)
top = lab[lab["level"] == "top"]["tokens"].sum()
print("전술(top) 13개만 넣을 때:", top, "토큰")

In [ ]:
# 예시까지 붙이면 얼마나 쓰는지 계산 (로그 문장 1개 길이는 실제 윈도우 문장으로 바꿔서 재보기)
log_sentence = "AWS CloudTrail session by IAMUser: 5 API calls, 0 failed, 80% read-only. Calls: iam:ListUsers x3, iam:GetAccountAuthorizationDetails x1, sts:GetCallerIdentity x1"
log_tok = len(tok(log_sentence)["input_ids"])
ex_tok = log_tok + 10   # 예시 1개 = 로그 문장 + 정답 라벨 이름 정도

print("로그 문장 1개:", log_tok, "토큰")
for k in [1, 3, 5, 8, 16]:
    total_top = top + log_tok + k * 13 * ex_tok          # 전술 13개 x 라벨당 k개 예시
    total_all = lab["tokens"].sum() + log_tok + k * 13 * ex_tok
    print(f"k={k:2d}  전술13개+예시 {total_top:6d} ({'OK' if total_top <= LIMIT else '초과'})"
          f"  |  라벨 전체+예시 {total_all:6d} ({'OK' if total_all <= LIMIT else '초과'})")
print("\n-> 퓨샷 파인튜닝은 시험 때 예시를 안 붙이므로 '예시' 부분이 0이 된다")